In [26]:
# IMPORTS
from rldb.utils import *
import torch
import numpy as np
from egomimic.utils.egomimicUtils import CameraTransforms, draw_actions
import torchvision.io as io
import os

In [27]:
# Load dataset
root = "/nethome/rpunamiya6/cedar-dx/datasets/EgoBridge/fold_shirts/lerobot_6dof"
repo_id = "rpuns/aria_laundry_rl2"

episodes = [0, 1]
dataset = RLDBDataset(repo_id=repo_id, root=root, local_files_only=True, episodes=episodes, mode="sample")


Returning existing local_dir `/nethome/rpunamiya6/cedar-dx/datasets/EgoBridge/fold_shirts/lerobot_6dof` as remote repo cannot be accessed in `snapshot_download` (None).
Returning existing local_dir `/nethome/rpunamiya6/cedar-dx/datasets/EgoBridge/fold_shirts/lerobot_6dof` as remote repo cannot be accessed in `snapshot_download` (None).
Returning existing local_dir `/nethome/rpunamiya6/cedar-dx/datasets/EgoBridge/fold_shirts/lerobot_6dof` as remote repo cannot be accessed in `snapshot_download` (None).


In [28]:
# Get metadata
print(dataset.meta.info["features"])

image_key = "observations.images.front_img_1"
actions_key = "actions_cartesian"

{'observations.state.ee_pose': {'dtype': 'float64', 'shape': (12,), 'names': ['dim_0']}, 'observations.images.front_img_1': {'dtype': 'image', 'shape': (3, 480, 640), 'names': ['channel', 'height', 'width']}, 'actions_cartesian': {'dtype': 'prestacked_float64', 'shape': (100, 12), 'names': ['chunk_length', 'action_dim']}, 'metadata.embodiment': {'dtype': 'int32', 'shape': (1,), 'names': ['dim_0']}, 'timestamp': {'dtype': 'float32', 'shape': (1,), 'names': None}, 'frame_index': {'dtype': 'int64', 'shape': (1,), 'names': None}, 'episode_index': {'dtype': 'int64', 'shape': (1,), 'names': None}, 'index': {'dtype': 'int64', 'shape': (1,), 'names': None}, 'task_index': {'dtype': 'int64', 'shape': (1,), 'names': None}}


In [29]:
# Make data_loader
data_loader = torch.utils.data.DataLoader(dataset,
                                          batch_size=1,
                                          shuffle=False)

In [30]:
camera_transforms = CameraTransforms(intrinsics_key="base", extrinsics_key="ariaJun7")

In [31]:
def visualize_actions(ims, actions, extrinsics, intrinsics, arm="both"):
    for b in range(ims.shape[0]):
        if actions.shape[-1] == 7 or actions.shape[-1] == 14:
            ac_type = "joints"
        elif actions.shape[-1] == 3 or actions.shape[-1] == 6:
            ac_type = "xyz"
        else:
            raise ValueError(f"Unknown action type with shape {actions.shape}")

        arm = "right" if actions.shape[-1] == 7 or actions.shape[-1] == 3 else "both"
        ims[b] = draw_actions(ims[b], ac_type, "Purples", actions[b], extrinsics, intrinsics, arm=arm)

    return ims


In [35]:
import os
import torch
import torchvision.io as io

save_dir = "./visualization/"
os.makedirs(save_dir, exist_ok=True)

num_batches = 200
fps = 30

skip_frames = 0

all_frames = []  # store frames from all batches

for i, data in enumerate(data_loader):
    if i <= skip_frames:
        continue
    if i - skip_frames >= num_batches:
        break

    ims = (data[image_key].permute(0, 2, 3, 1).cpu().numpy() * 255.0).astype(np.uint8)
    actions_left = data[actions_key].cpu().numpy()[..., :3]
    actions_right = data[actions_key].cpu().numpy()[..., 6:9]
    
    actions = np.concatenate([actions_left, actions_right], axis=-1)

    ims_viz = visualize_actions(ims, actions, camera_transforms.extrinsics, camera_transforms.intrinsics)
    # ims_viz: list of HxWx3 uint8 RGB

    for im in ims_viz:
        all_frames.append(torch.from_numpy(im).to(torch.uint8))

# Stack into T×H×W×C tensor
video_tensor = torch.stack(all_frames, dim=0)

# (Optional) make dimensions even
H, W = video_tensor.shape[1], video_tensor.shape[2]
if H % 2 != 0:
    video_tensor = video_tensor[:, :H-1, :, :]
if W % 2 != 0:
    video_tensor = video_tensor[:, :, :W-1, :]

video_path = os.path.join(save_dir, "all_batches_human.mp4")
io.write_video(video_path, video_tensor, fps=fps, video_codec="h264")
print(f"Saved combined video to {video_path}")


Saved combined video to ./visualization/all_batches_human.mp4
